<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
 }

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>


# Introduction to Docker

This notebook introduces Docker and the specific features used in Duckietown. For a broader overview, consult [Docker’s official Get Started guide](https://docs.docker.com/get-started/overview/), and check out the [Duckietown Introduction to Containerization lecture](https://app-na1.hubspotdocuments.com/documents/8795519/view/717428738?accessId=475a8d) for additional insight.

## Why Docker?

In other learning experiences, e.g., the [Linux and Networking LX](https://github.com/duckietown/lx-dd-linux-and-networking), you learned how to run programs, inspect processes, and connect to a Duckiedrone. Running your own robotics software introduces another challenge: the program needs the right environment.

For example, a camera-processing script may depend on a particular Python version and several image-processing libraries. Copying the script from your base station to a robot does not copy those dependencies. Installing them directly on the robot can also create conflicts with software already there.

Docker provides operating-system-level virtualization, commonly called _containerization_. Containers isolate user-space instances so that programs perceive a dedicated environment while sharing the host kernel. This approach yields lightweight, reproducible deployments—ideal for distributed robotics platforms such as the Raspberry Pi used in Duckietown.

In other words, containerization allows the existence of multiple isolated user space instances called containers. These containers may look like real computers from the point of view of programs running in them.

A computer program running on an ordinary operating system can see all resources available to the system, e.g., network devices, CPU, and RAM. However, programs running inside a container can only see the container's resources. Resources assigned to the container become available to all processes that live inside that container.

This notebook explains what a container is, how images define its software environment, and how Docker manages it. These distinctions will help you understand what happens when you start a container in the following activities.

## What is a container?

Consider two applications that process Duckiedrone camera images. One requires an older version of an image-processing library; the other requires a newer version. Installing both into the same software environment may be difficult.

With containers, each application can have its own files, libraries, and installed tools. Both can run on the same computer without requiring the same library version.

From inside a container, a shell can look familiar: commands such as `ls`, `cd`, and `cat` still work. However, the filesystem you explore belongs to the container’s environment. Installing a Python library there does not normally install it into the host’s Python environment.

The **host** is the computer providing the resources on which the container runs. Containers use its processor, memory, and kernel; they do not provide their own additional computing capacity.

Docker uses Linux **namespaces** to give processes separate views of resources such as filesystem mounts, process identifiers, and networking. The degree of separation depends on the container’s configuration. For example, a directory can be deliberately shared between the host and a container. [Docker’s underlying technology](https://docs.docker.com/get-started/docker-overview/#the-underlying-technology)

## How is a container different from a virtual machine?

A virtual machine also provides an environment in which to run software. The important difference is what that environment includes.

A **virtual machine (VM)** runs a guest operating system with its own kernel. Linux containers share the kernel of their Linux host, while keeping their application environments separate.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
Virtual machines                      Linux containers

+----------------+ +----------------+  +----------------+ +----------------+
| Application A  | | Application B  |  | Application A  | | Application B  |
| Libraries      | | Libraries      |  | Libraries      | | Libraries      |
| Guest kernel   | | Guest kernel   |  +----------------+ +----------------+
+----------------+ +----------------+  |       Shared Linux kernel        |
|       Virtualization layer        |  +----------------------------------+
+-----------------------------------+
  </pre>
  <figcaption>
    Figure 1: Each VM has a guest kernel; Linux containers share one.
  </figcaption>
</figure>

Sharing a kernel generally reduces the overhead of running separate environments. This is useful on a robot, where memory and processing power must also support sensing, communication, and control.

An Ubuntu container can therefore provide Ubuntu command-line tools and libraries without booting a separate Ubuntu kernel. On systems such as macOS, Linux containers run inside a Linux VM; they share that VM’s kernel. [What is a container?](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/)

## How does Docker work? Docker images, layers, and containers

Docker uses three main entities: images → layers → containers. Containers are built from images, which are made of layers.

- Image: build-time artifact

- Layer: incremental filesystem change inside an image

- Container: run-time instance of an image

Docker images are _build-time_ artifacts while Docker containers are _run-time_ constructs.

In other words, a Docker image is static, like a `.zip` or `.iso` file. A container, on the other hand, starts from a static image but as it is used, files and configurations might change.

Docker images are built from layers. The initial, or lowest-level, layer is the _base layer_. Typically, this is a stripped-down version of an OS. For example, many Docker images in Duckietown run `ubuntu:18.04` or `ubuntu:22.04` as a base.

Each layer on top of the base layer represents a change to the layers below. Docker translates this sequence of changes to a file system that the container can then use.

Small changes to a single file typically only modify one layer. Therefore, when Docker attempts to pull the new version, it will need to download and store only the modified layer, saving space, time, and bandwidth.

## Images define the environment; containers use it

Before starting our camera-processing application, Docker needs a description of the files and software it should make available.

An **image** is the read-only template used to create a container. For our example, it could contain Python, the image-processing library, and the application code.

A **container** is an instance created from that image. One image can be used to create several containers, each with its own configuration and writable filesystem layer.

For example, we could use the same image to create:

- `lx-docker-camera-a`, which processes a recording from one experiment.
- `lx-docker-camera-b`, which processes a recording from another experiment.

Both begin with the same application and dependencies. A file written into the writable layer of the first container does not automatically appear in the second or change the original image.

Stopping a container preserves its writable layer. Removing the container deletes that layer. Later activities introduce storage that keeps experiment results independently of a container’s lifetime. [Docker storage](https://docs.docker.com/engine/storage/)

### How is an image built?

A **Dockerfile** records the instructions for building an image. The build could start from an existing Python image, install the required library, and copy in a processing script.

These filesystem changed would be stored in **layers**:

<figure id="figure-2" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
Illustrative image contents

+---------------------------------------+
| Camera-processing script              |
+---------------------------------------+
| Additional image-processing library   |
+---------------------------------------+
| Base image: Python and supporting     |
| Linux tools and libraries             |
+---------------------------------------+
  </pre>
  <figcaption>
    Figure 2: Image layers store filesystem changes over the base OS layer
  </figcaption>
</figure>

The base image may itself contain several layers. Docker combines them into the filesystem visible inside a container and can reuse layers already available locally. This reduces repeated downloads and storage when images share software. For additional detail, check out Docker's "[Understanding image layers](https://docs.docker.com/get-started/docker-concepts/building-images/understanding-image-layers/)".

## Registries (like DockerHub) make images available to other computers

Suppose we have built a processing image, and a teammate wants to use it.

A **registry** stores and distributes images. As with Git and GitHub, the most popular Docker register, and the one used in Duckietown, is [Docker Hub](https://hub.docker.com/). **Pushing** uploads an image to a registry; **pulling** downloads it.

The workflow is:

```text
Build an image → Push it to a registry → Pull it elsewhere → Create a container
```

Pulling an image does not run the application. It makes the image available so that Docker can create containers from it.

Packaging the software also does not remove hardware requirements. An image must support the target computer’s processor architecture, and an application that reads a physical camera still needs access to that device.

### Docker Hub and Duckietown

Docker Hub organizes images as:

```text
[repository/]image[:tag]
```

The parts `repository` and `tag` are optional, and default to `library` (indicating Docker official images) and `latest` (special tag always pointing to the _latest_ version of an image).

For example, in the Duckietown Docker image:

```text
duckietown/dt-core:ente-arm64v8
```

- The repository name is `duckietown`.

- The image name is `dt-core`.

- The tag is `ente-arm64v8`.

The tag specifies the Duckietown software distribution that the image contains, i.e., `ente`, and the CPU architecture that this image is targeting, i.e., `arm64v8`. We will talk about different CPU architectures and why they need to be part of the Docker image tag in [](basic-docker-arch).

All Duckietown-related images are hosted in the `duckietown` Docker Hub repository, although they can serve very different applications.

## Docker Engine manages images and containers

When you ask Docker to start a container, two components cooperate:

- The **Docker client**, usually the `docker` command-line interface, sends your request.
- The **Docker daemon**, `dockerd`, carries it out and manages resources such as images, containers, networks, and storage volumes.

They communicate through an **Application Programming Interface (API)**. Together, these components form Docker Engine’s client-server architecture. [Docker Engine](https://docs.docker.com/engine/)

The client and daemon can run on different computers. For example, a client on your base station can connect to a daemon on the Duckiedrone. A request to start a container then starts it on the robot.

```text
Base station                          Duckiedrone
Docker client ─── API request ───────> Docker daemon
                                            |
                                            v
                                      Starts container
```

A **Docker context** stores the connection settings used to select a daemon. Therefore, the terminal in which you type a command does not, by itself, identify where Docker will perform the action. [Docker contexts](https://docs.docker.com/engine/manage-resources/contexts/)

The following activities establish a local practice environment before creating resources. We will use names beginning with `lx-docker-` to identify exercise resources and check the daemon connection before making changes.

## Further reading

The [Duckietown Docker introduction](https://docs.duckietown.com/ente/duckietown-manual/70-developer-manual/basics/development/developer-basics-docker.html) connects these concepts to the tools and workflows used in Duckietown. Docker’s [What is Docker?](https://docs.docker.com/get-started/docker-overview/) guide provides a broader platform overview.

## Checkpoint

Before running the self-check, consider these questions:

1. Why can two containers use different Python libraries while sharing the same Linux kernel?
2. You create two containers from one image. Does writing a file in one container’s writable layer change the image?
3. Your Docker client runs on the base station and connects to the Duckiedrone’s daemon. Where does a new container run?

Run the self-check in the next cell. Write or select a response before revealing the answer.



---



## How to interface with Docker? The engine is client-server software

Docker Engine uses a client-server architecture. The `docker` command-line interface (CLI) sends Application Programming Interface (API) requests to the long-running `dockerd` daemon. The daemon creates and manages Docker objects, including images, containers, networks, and volumes. The client and daemon can run on the same computer or on different computers. The relationship is summarized in [Figure 1](#figure-1).

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    Docker CLI or Compose client
      |
      | Docker API
      v
    Docker daemon (`dockerd`) <-> image registry
      |
      v
    images, containers, networks, and volumes
  </pre>
  <figcaption>
    Figure 1: Docker client, daemon, registry, and managed objects.
  </figcaption>
</figure>

A Docker context tells the client which daemon connection settings to use. Later notebooks inspect contexts before any modifying command, because the terminal window where you type a command does not by itself identify the daemon that receives it.

## Images, containers, and registries

An image is a read-only template used to create containers. A Dockerfile describes the image build steps, and its instructions produce ordered filesystem layers. A container is a runnable instance of an image with configuration such as storage and network settings. Removing a container discards changes in its writable layer unless those changes were stored in persistent storage such as a volume.

A registry stores and distributes images. Pulling obtains an image from a registry, while pushing publishes an image to one. An image and a container are different objects: one image can create multiple independently configured containers.

## Containers and the host

A Linux container is not a virtual machine. Docker uses Linux kernel features, including namespaces, to isolate a container's view of resources. A namespace gives a process its own view of processes, networking, and filesystem mounts. Containers on a Linux host share that host's kernel; a virtual machine instead runs a separate guest operating system and kernel. Isolation depends on the configuration given to a container, so a container is not an absolute security boundary.

This learning experience (LX) uses only learner-owned local resources with names beginning `lx-docker-`. Its later notebooks distinguish a local practice daemon from a Duckiedrone platform daemon. Understanding the client, daemon, and object model is the basis for keeping those targets separate.

## Further reading

Consult Docker's [What is Docker?](https://docs.docker.com/get-started/docker-overview/) guide for the platform, its client-server architecture, registries, images, containers, and Linux kernel technology. The [Docker Engine documentation](https://docs.docker.com/engine/) provides more detail on the daemon, APIs, CLI, and Docker objects.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.

In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
